# CuPy example

In this example we want to do two things. 1) we want to calculate the distances between two datasets and then do an fourier transformation based on the resulting distance distribution. Here we want to compare the runtime between numpy and cupy to see the benefit of GPU offloading and how simple it is to translate existing numpy code to cupy.

First load some libraries - in particular numpy and timeit to measure the time required by the function calls.

In [ ]:
import numpy as np
import timeit
import matplotlib.pyplot as plt

!nvidia-smi # check if we have a GPU availiable

Generate some random data to work with:

In [ ]:
npart_data = 1000
Nbins = 10000

Data1 = np.random.rand(npart_data,2)
Data2 = np.random.rand(npart_data,2)
print(f"Running with {Data1.shape[0]} x {Data2.shape[0]} particles and sorting in {Nbins} bins", flush=True)
bins = np.linspace(0, 1.5, Nbins+1)

## Distance distribution

Lets define some functions using numpy to calculate the distances between two datasets using numpy array operations

In [ ]:
def distances_CPU_numpy(Data1, Data2, bins):
    dist_hist = get_distances_CPU_numpy(Data1, Data2, bins)
    return dist_hist

def get_distances_CPU_numpy(Data1, Data2, bins):
    # Compute pairwise distances
    dx = Data1[:, 0][:, np.newaxis] - Data2[:, 0][np.newaxis, :]
    dy = Data1[:, 1][:, np.newaxis] - Data2[:, 1][np.newaxis, :]
    dist = np.sqrt(dx**2 + dy**2)

    # Flatten the distance matrix and compute histogram
    dist_hist = np.histogram(dist.ravel(), bins=bins)[0]
    return dist_hist

Now calculate the distribution and generate a simple histogram:

In [ ]:
# numpy CPU
CPU_comp = distances_CPU_numpy(Data1, Data2, bins)

plt.bar(bins[:-1],CPU_comp, width=np.diff(bins), align="edge")
plt.ylabel("Counts")
plt.xlabel("Distance")
plt.show()

Now lets have a look at cupy!

In [ ]:
import cupy as cp

def distances_GPU_cupy(Data1, Data2, bins):
    # Convert input arrays to CuPy
    Data1 = cp.asarray(Data1)
    Data2 = cp.asarray(Data2)
    bins = cp.asarray(bins)
    dist_hist = get_distances_GPU_cupy(Data1, Data2, bins)
    return dist_hist.get()

def get_distances_GPU_cupy(Data1, Data2, bins):
    # Calculate distances in a vectorized manner
    dx = Data1[:, 0][:, cp.newaxis] - Data2[:, 0][cp.newaxis, :]
    dy = Data1[:, 1][:, cp.newaxis] - Data2[:, 1][cp.newaxis, :]
    dist = cp.sqrt(dx**2 + dy**2)

    # Use cp.histogram for GPU histogramming
    dist_hist = cp.histogram(dist, bins)[0]
    return dist_hist

Note the difference to numpy: mainly just replacing the alias. One aspect to take care of is the data movement to and back from the device. Explicit declaration as cupy arrays and .get() function to port back to host. Alternatively oen can use cp.asnumpy().

Now lets do the calculation:

In [ ]:
GPU_comp = distances_GPU_cupy(Data1, Data2, bins)

plt.bar(bins[:-1],GPU_comp, width=np.diff(bins), align="edge")
plt.ylabel("Counts")
plt.xlabel("Distance")
plt.show()

Lets compare the two results in detail:

In [ ]:
np.array_equal(GPU_comp, CPU_comp)

Great! So now lets do some time comparisons:

In [ ]:
# now do some time comparisons - run each function n times
runs = 10
total_time = timeit.timeit(lambda: distances_CPU_numpy(Data1, Data2, bins), number=runs)
mean_time_CPU = total_time / runs
print(f"CPU:\nTotal time for {runs} runs: {total_time:.6f} seconds")
print(f"Mean time per run: {mean_time_CPU:.6f} seconds")

total_time = timeit.timeit(lambda: distances_GPU_cupy(Data1, Data2, bins), number=runs)
mean_time_GPU = total_time / runs
print(f"GPU cupy:\nTotal time for {runs} runs: {total_time:.6f} seconds")
print(f"Mean time per run: {mean_time_GPU:.6f} seconds")

print(f"\nRuntime ratio (CPU/GPU): {mean_time_CPU / mean_time_GPU:.2f}×")

### CuPy Kernel
Lets use a custom cude kernel to calculate the distances and do the binning

In [ ]:
get_distances_kernel = cp.RawKernel(
    r"""
extern "C" __global__
void get_distances_kernel(const double* __restrict__ Data1,
                          const double* __restrict__ Data2,
                          const double* __restrict__ bins,
                          const int nData1, const int nData2,
                          const int nbins,
                          int* __restrict__ hist)
{
    long long idx = (long long)blockDim.x * blockIdx.x + threadIdx.x;
    long long total = (long long)nData1 * (long long)nData2;

    // Skip out-of-range threads
    if (idx >= total)
        return;

    int a_idx = idx / nData2;
    int b_idx = idx % nData2;

    // Each data point has 2 coordinates (x, y)
    double dx = Data1[a_idx * 2 + 0] - Data2[b_idx * 2 + 0];
    double dy = Data1[a_idx * 2 + 1] - Data2[b_idx * 2 + 1];
    double dist = sqrt(dx * dx + dy * dy);

    // Binary search for bin index
    int left = 0, right = nbins - 1, mid;
    while (left < right)
    {
        mid = (left + right) >> 1;
        if (dist >= bins[mid + 1])
            {
            left = mid + 1;
            }
        else
            {
            right = mid;
            }
    }

    // Match NumPy's np.histogram rule:
    // include right edge of last bin
    if (dist >= bins[0] && (dist < bins[nbins] || (left == nbins - 1 && dist == bins[nbins])))
    {
        atomicAdd(&hist[left], 1);
    }
}
""",
    "get_distances_kernel",
)

In [ ]:
def distances_cupy_kernel(Data1, Data2, bins):
    # Number of points
    nData1, nData2 = Data1.shape[0], Data2.shape[0]
    nbins = len(bins) - 1

    # Move data to device
    GPUbins  = cp.asarray(bins, dtype=cp.float64)
    GPUData1 = cp.asarray(Data1)
    GPUData2 = cp.asarray(Data2)
    
    # Allocate histogram on GPU
    hist = cp.zeros(nbins, dtype=cp.int32)

    # Launch parameters
    threads = 256
    blocks = (
        nData1 * nData2 + threads - 1
    ) // threads  # not checked if this exceeds integer limits
    
    # Launch the kernel and make sure that all scalar kernel arguments are NumPy types
    get_distances_kernel(
        (blocks,),
        (threads,),
        (
            GPUData1.astype(cp.float64).ravel(),
            GPUData2.astype(cp.float64).ravel(),
            GPUbins.astype(cp.float64),
            np.int32(nData1),
            np.int32(nData2),
            np.int32(nbins),
            hist,
        ),
    )
    return hist.get()

Run kernel and check if results:

In [ ]:
GPU_kernel_comp = distances_cupy_kernel(Data1, Data2, bins)
np.array_equal(GPU_comp, GPU_kernel_comp)

Now lets time the kernel:

In [ ]:
total_time = timeit.timeit(lambda: distances_cupy_kernel(Data1, Data2, bins), number=runs)
mean_time_GPU_kernel = total_time / runs
print(f"GPU cupy kernel:\nTotal time for {runs} runs: {total_time:.6f} seconds")
print(f"Mean time per run: {mean_time_GPU:.6f} seconds")

print(f"\nRuntime ratio (CPU/GPU kernel): {mean_time_CPU / mean_time_GPU_kernel:.2f}×")
print(f"\nRuntime ratio (GPU/GPU kernel): {mean_time_GPU / mean_time_GPU_kernel:.2f}×")

## Let's compare the fourier transformation

Again lets define some function in numpy and cupy:

In [ ]:
from scipy import fft

# numpy implementation
def get_powerspec_CPU(dist_hist, bins):
    fft_result, frequencies = fourier_transform_scipy(np.asarray(dist_hist), bins)
    return fft_result, frequencies

def fourier_transform_scipy(correlation_array, bins):
    fft_result = fft.fft(correlation_array)
    bin_width = np.diff(bins)
    frequencies = fft.fftfreq(len(correlation_array), d=bin_width)
    return fft_result, frequencies

# cupy implementation
def get_powerspec_GPU(dist_hist, bins):
    fft_result, frequencies = fourier_transform_cupy(cp.asarray(dist_hist), bins)
    return cp.asnumpy(fft_result), cp.asnumpy(frequencies) # use cp.asnumpy this time

def fourier_transform_cupy(correlation_array, bins):
    fft_result = cp.fft.fft(correlation_array)
    bin_width = cp.diff(bins)
    frequencies = cp.fft.fftfreq(len(correlation_array), d=bin_width)
    return fft_result, frequencies

Again see how minimal the differences are?

Now lets compare their output:

In [ ]:
# CPU
power_spec_CPU, frequencies_CPU = get_powerspec_CPU(CPU_comp, bins)

# GPU
power_spec_GPU, frequencies_GPU = get_powerspec_GPU(GPU_comp, bins)

# do comparison - use allclose due to floating point precission
print("The fft_result are equal? ", np.allclose(power_spec_CPU, power_spec_GPU))
print("The frequencies are equal? ", np.allclose(frequencies_CPU, frequencies_GPU))

So lets compare the runtimes:

In [ ]:
# now do some time comparisons - run each function n times
runs = 10
total_time = timeit.timeit(lambda: get_powerspec_CPU(GPU_comp, bins), number=runs)
mean_time_CPU = total_time / runs
print(f"CPU:\nTotal time for {runs} runs: {total_time:.6f} seconds")
print(f"Mean time per run: {mean_time_CPU:.6f} seconds")

total_time = timeit.timeit(lambda: get_powerspec_GPU(GPU_comp, bins), number=runs)
mean_time_GPU = total_time / runs
print(f"GPU cupy:\nTotal time for {runs} runs: {total_time:.6f} seconds")
print(f"Mean time per run: {mean_time_GPU:.6f} seconds")

print(f"\nRuntime ratio (CPU/GPU): {mean_time_CPU / mean_time_GPU:.2f}×")

Why is the GPU slower in this case? -> maybe workload is to small?
Lets try a larger dataset!

In [ ]:
# new len of bins
len_new_hist = int(1e6)

# create array with random int values
random_hist = np.random.randint(low=0, high=1000, size=len_new_hist)
bins_new = np.linspace(0, 1.5, len_new_hist+1)

In [ ]:
# now do some time comparisons - run each function n times
runs = 10
total_time = timeit.timeit(lambda: get_powerspec_CPU(random_hist, bins_new), number=runs)
mean_time_CPU = total_time / runs
print(f"CPU:\nTotal time for {runs} runs: {total_time:.6f} seconds")
print(f"Mean time per run: {mean_time_CPU:.6f} seconds")

total_time = timeit.timeit(lambda: get_powerspec_GPU(random_hist, bins_new), number=runs)
mean_time_GPU = total_time / runs
print(f"GPU cupy:\nTotal time for {runs} runs: {total_time:.6f} seconds")
print(f"Mean time per run: {mean_time_GPU:.6f} seconds")

print(f"\nRuntime ratio (CPU/GPU): {mean_time_CPU / mean_time_GPU:.2f}×")